# Prework - Čištění dat (Cleaning Data: Heart Disease Dataset)

Dataset `heart_data.csv` obsahuje diagnostické lékařské parametry pacientů pro určení diagnózy srdečního onemocnění (**ahd** - Angina / Heart Disease: `yes` / `no`).

### Seznam proměnných v datasetu:
- `age`: věk pacienta
- `sex`: pohlaví pacienta (`Male` / `Female`)
- `chestpain`: typ bolesti na hrudi (`typical`, `asymptomatic`, `nonanginal`, `nontypical`)
- `trestbps` (v datasetu `restbp`): klidový krevní tlak (v mmHg)
- `chol`: hladina cholesterolu v krvi (v mg/dl)
- `fbs`: hladina cukru v krvi nalačno (> 120 mg/dl $\rightarrow$ `Yes`, jinak `No`)
- `restecg`: výsledky klidového EKG vyšetření
- `minhr`: minimální naměřená tepová frekvence
- `maxhr`: maximální naměřená tepová frekvence
- `exang`: cvičením vyvolaná angina pectoris (`Yes` / `No`)
- `oldpeak`: deprese ST segmentu vyvolaná zátěží vzhledem ke klidu
- `slope`: sklon ST segmentu
- `ca`: počet velkých cév barvených fluoroskopií (0–3)
- `thal`: thalasémie (genetické onemocnění krve: `normal`, `fixed`, `reversable`)
- `ahd`: diagnóza srdečního onemocnění (cílová proměnná: `Yes` / `No`)

## 1. a 2. Načtení dat a zobrazení posledních 15 pozorování

In [1]:
import os
import pandas as pd

# Načtení dat
data_path = os.path.join("data", "heart_data.csv")
df = pd.read_csv(data_path)

print(f"Rozměry načtených dat: {df.shape[0]} řádků, {df.shape[1]} sloupců")
# Zobrazení posledních 15 pozorování
df.tail(15)

Rozměry načtených dat: 313 řádků, 15 sloupců


,age,sex,chestpain,restbp,chol,fbs,restecg,minhr,maxhr,exang,oldpeak,slope,ca,thal,ahd
298,45,Male,typical,110,264.0,No,0,NaN,132,No,1.2,2,0.0,reversable,Yes
299,68,Male,asymptomatic,144,193.0,Yes,0,NaN,141,No,3.4,2,2.0,reversable,Yes
300,57,Male,asymptomatic,130,131.0,No,0,NaN,115,Yes,1.2,2,1.0,reversable,Yes
301,57,Female,nontypical,130,236.0,No,2,NaN,174,No,0.0,2,1.0,normal,Yes
302,38,Male,nonanginal,138,175.0,No,0,NaN,173,No,0.0,1,NaN,normal,No
303,57,Male,asymptomatic,140,192.0,No,0,NaN,148,No,0.4,2,0.0,fixed,No
304,56,Female,nontypical,140,294.0,No,2,NaN,153,No,1.3,2,0.0,normal,No
305,56,Male,nonanginal,130,256.0,Yes,2,NaN,142,Yes,0.6,2,1.0,fixed,Yes
306,44,Male,nontypical,120,263.0,No,0,NaN,173,No,0.0,1,0.0,reversable,No
307,52,Male,nonanginal,172,199.0,Yes,0,NaN,162,No,0.5,1,0.0,reversable,No


## 3. Odstranění redundantních proměnných

Prozkoumáme výskyt prázdných hodnot a identifikujeme sloupce, které nenesou žádnou informaci. Sloupec `minhr` obsahuje 100 % chybějících hodnot (`NaN` ve všech 313 řádcích) a je pro model zcela nepoužitelný.

In [2]:
# Analýza chybějících hodnot ve všech sloupcích
print(df.isnull().sum())

# Odstranění sloupce 'minhr'
df = df.drop(columns=["minhr"])
print(f"\nNový tvar datasetu po odstranění 'minhr': {df.shape}")

age            0
sex            0
chestpain      0
restbp         0
chol           2
fbs            0
restecg        0
minhr        313
maxhr          0
exang          0
oldpeak        0
slope          0
ca             4
thal           2
ahd            0
dtype: int64

Nový tvar datasetu po odstranění 'minhr': (313, 14)


## 4. Detekce a odstranění duplicit

Pomocí `.duplicated().sum()` zjistíme počet duplicitních záznamů a pomocí `.drop_duplicates()` je odstraníme.

In [3]:
num_dupes = df.duplicated().sum()
print(f"Počet nalezených duplicitních řádků: {num_dupes}")

if num_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Počet unikátních řádků po odstranění duplicit: {len(df)}")

Počet nalezených duplicitních řádků: 10
Počet unikátních řádků po odstranění duplicit: 303


## 5. Ošetření chybějících hodnot (Imputace)

Dle zadání doplníme chybějící hodnoty v numerických proměnných jejich **mediánem** (`.median()`). V kategorickém sloupci `thal` doplníme chybějící hodnoty nejčastější hodnotou (**modus**).

In [4]:
print("Chybějící hodnoty před imputací:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Doplnění numerických sloupců mediánem
for col in ["chol", "ca"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"Sloupec '{col}' doplněn mediánem: {median_value}")

# Doplnění kategorického sloupce thal modem
thal_mode = df["thal"].mode()[0]
df["thal"] = df["thal"].fillna(thal_mode)
print(f"Sloupec 'thal' doplněn modem: '{thal_mode}'")

print(f"\nCelkový počet zbývajících chybějících hodnot: {df.isnull().sum().sum()}")

Chybějící hodnoty před imputací:
chol    2
ca      4
thal    2
dtype: int64
Sloupec 'chol' doplněn mediánem: 240.0
Sloupec 'ca' doplněn mediánem: 0.0
Sloupec 'thal' doplněn modem: 'normal'

Celkový počet zbývajících chybějících hodnot: 0


## 6. Kontrola a oprava chyb v kategorických proměnných

V textových polích se často vyskytují nestejné velikosti písmen (např. `Male`/`male`, `Yes`/`yes`, `FIXED`/`fixed`) a překlepy:
- `chestpain`: překlepy jako `nontypica`, `asymptomaticc`, `nontypiccal`
- `thal`: překlep `reversabble`

In [7]:
cat_cols = ["sex", "chestpain", "fbs", "exang", "thal", "ahd"]

# Sjednocení na malá písmena
for col in cat_cols:
    df[col] = df[col].astype(str).str.lower()

# Oprava překlepů v 'chestpain'
chestpain_fixes = {
    "nontypica": "nontypical",
    "asymptomaticc": "asymptomatic",
    "nontypiccal": "nontypical"
}
df["chestpain"] = df["chestpain"].replace(chestpain_fixes)

# Oprava překlepu v 'thal'
thal_fixes = {"reversabble": "reversable"}
df["thal"] = df["thal"].replace(thal_fixes)

# Kontrola unikátních hodnot po opravě
for col in cat_cols:
    print(f"{col:10s}: {sorted(df[col].unique())}")

sex       : ['female', 'male']
chestpain : ['asymptomatic', 'nonanginal', 'nontypical', 'typical']
fbs       : ['no', 'yes']
exang     : ['no', 'yes']
thal      : ['fixed', 'normal', 'reversable']
ahd       : ['no', 'yes']


## 7. Uložení vyčištěného datasetu do .csv bez indexu

Vyčištěná data uložíme jako `heart_data_exercise_1.csv` s argumentem `index=False`.

In [8]:
output_path = os.path.join("data", "heart_data_exercise_1.csv")
df.to_csv(output_path, index=False)
print(f"Vyčištěný dataset uložen do '{output_path}'")
print(f"Finální rozměry: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()

Vyčištěný dataset uložen do 'data\heart_data_exercise_1.csv'
Finální rozměry: 303 řádků, 14 sloupců


,age,sex,chestpain,restbp,chol,fbs,restecg,maxhr,exang,oldpeak,slope,ca,thal,ahd
0,63,male,typical,145,233.0,yes,2,150,no,2.3,3,0.0,fixed,no
1,67,male,asymptomatic,160,286.0,no,2,108,yes,1.5,2,3.0,normal,yes
2,67,male,asymptomatic,120,229.0,no,2,129,yes,2.6,2,2.0,reversable,yes
3,37,male,nonanginal,130,240.0,no,0,187,no,3.5,3,0.0,reversable,no
4,41,female,nontypical,130,204.0,no,2,172,no,1.4,1,0.0,fixed,no
